In [0]:
from pyspark.sql import functions as F

In [0]:
base_path = "/Volumes/olist_data/raw/olist_files/"
customer = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_customers_dataset.csv")
)
geolocation = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_geolocation_dataset.csv")
)
order_items = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_order_items_dataset.csv")
)
order_payments = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_order_payments_dataset.csv")
)
order_reviews = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("unescapedQuoteHandling", "BACK_TO_DELIMITER")
    .load(base_path+"bronze_order_reviews_dataset.csv")
)
orders = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_orders_dataset.csv")
)
products = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_products_dataset.csv")
)
product_category = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path + "bronze_product_category_name_translation.csv")
)
sellers = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path+"bronze_sellers_dataset.csv")
)

In [0]:
print("Customer")
display(customer.limit(10))
print("Geolocation")
display(geolocation.limit(10))
print("Order_items")
display(order_items.limit(10))
print("Order_payments")
display(order_payments.limit(10))
print("Order_reviews")
display(order_reviews.limit(10))
print("Orders")
display(orders.limit(10))
print("Products")
display(products.limit(10))
print("Product_category")
display(product_category.limit(10))
print("Sellers")
display(sellers)

In [0]:
print("Customer : " + str(customer.count()))
print("Geolocation : " + str(geolocation.count()))
print("Order_items : " + str(order_items.count()))
print("Order_payments : " + str(order_payments.count()))
print("Order_reviews : " + str(order_reviews.count()))
print("Orders : " + str(orders.count()))
print("Products : " + str(products.count()))
print("Product_category : " + str(product_category.count()))
print("Sellers : "+ str(sellers.count()))


In [0]:
print(f"Customer : {len(customer.columns)} | columns : "+ str(customer.columns))
print("\n")
print(f"Geolocation : {len(geolocation.columns)} | columns : "+ str(geolocation.columns))
print("\n")
print(f"Order_items : {len(order_items.columns)} | columns : "+ str(order_items.columns))
print("\n")
print(f"Order_payments : {len(order_payments.columns)} | columns : "+ str(order_payments.columns))
print("\n")
print(f"Order_reviews : {len(order_reviews.columns)} | columns : "+ str(order_reviews.columns))
print("\n")
print(f"Orders : {len(orders.columns)} | columns : "+ str(orders.columns))
print("\n")
print(f"Products : {len(products.columns)} | columns : "+ str(products.columns))
print("\n")
print(f"Product_category : {len(product_category.columns)} | columns : "+ str(product_category.columns))
print("\n")
print(f"Sellers : {len(sellers.columns)} | columns : "+ str(sellers.columns))

In [0]:
for name, df in [
    ("Customer", customer),
    ("Geolocation", geolocation),
    ("Order_items", order_items),
    ("Order_payments", order_payments),
    ("Order_reviews", order_reviews),
    ("Orders", orders),
    ("Products", products),
    ("Product_category", product_category),
    ("Sellers", sellers),
]:
    print(f"{name}: {df.count()} rows x {len(df.columns)} cols")

In [0]:
customer.printSchema()
geolocation.printSchema()
order_items.printSchema()
order_payments.printSchema()
order_reviews.printSchema()
orders.printSchema()
products.printSchema()
product_category.printSchema()
sellers.printSchema()

In [0]:
print("Customer")
display(customer.describe())
print("Geolocation")
display(geolocation.describe())
print("Order_items")
display(order_items.describe())
print("Order_payments")
display(order_payments.describe())
print("Order_reviews")
display(order_reviews.describe())
print("Orders")
display(orders.describe())
print("Products")
display(products.describe())
print("Product_category")
display(product_category.describe())
print("Sellers")
display(sellers.describe())

In [0]:
for df, name in [
    (customer, "Customer"),
    (geolocation, "Geolocation"),
    (order_items, "Order_items"),
    (order_payments, "Order_payments"),
    (order_reviews, "Order_reviews"),
    (orders, "Orders"),
    (products, "Products"),
    (product_category, "Product_category"),
    (sellers, "Sellers"),
]:
    print(f"{name}:")
    null_counts = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
    row = null_counts.collect()[0]
    for c in df.columns:
        print(f"  {c}: {row[c]}")
    print()


In [0]:
products.filter(
    F.col("product_category_name").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
).show()

In [0]:
products.filter(
    F.col("product_category_name").isNull() &
    F.col("product_name_lenght").isNull() &
    F.col("product_description_lenght").isNull() &
    F.col("product_photos_qty").isNull()
).count()

In [0]:
from pyspark.sql import functions as F

products = products.withColumn(
    "product_category_name",
    F.coalesce(
        F.col("product_category_name"),
        F.lit("unknown")
    )
)

In [0]:
orders.filter(
    F.col("order_delivered_customer_date").isNull()).groupBy("order_status").count().show()

In [0]:
display(
    orders.filter(
        (F.col("order_status") == "delivered") &
        (F.col("order_delivered_customer_date").isNull())
    )
)

In [0]:
from pyspark.sql import functions as F

orders = orders.withColumn(
    "delivery_date_missing_flag",
    F.when(
        (F.col("order_status") == "delivered") &
        (F.col("order_delivered_customer_date").isNull()),
        1
    ).otherwise(0)
)

In [0]:
for df, name in [
    (customer, "Customer"),
    (geolocation, "Geolocation"),
    (order_items, "Order_items"),
    (order_payments, "Order_payments"),
    (order_reviews, "Order_reviews"),
    (orders, "Orders"),
    (products, "Products"),
    (product_category, "Product_category"),
    (sellers, "Sellers"),
]:
    print(f"{name}:")
    null_counts = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
    row = null_counts.collect()[0]
    for c in df.columns:
        print(f"  {c}: {row[c]}")
    print()


In [0]:
for df, name in [
    (customer, "Customer"),
    (geolocation, "Geolocation"),
    (order_items, "Order_items"),
    (order_payments, "Order_payments"),
    (order_reviews, "Order_reviews"),
    (orders, "Orders"),
    (products, "Products"),
    (product_category, "Product_category"),
    (sellers, "Sellers"),
]:
    
    total_rows = df.count()

    print(f"{name}:")
    null_counts = df.agg(*[(F.sum(F.col(c).isNull().cast("int"))/total_rows*100).alias(c) for c in df.columns])
    row = null_counts.collect()[0]
    for c in df.columns:
        print(f"  {c}: {row[c]:.2f}%")
    print()

In [0]:
display(
    orders.select("order_status").distinct(),
    orders.groupBy("order_status").count()
)

In [0]:
order_items.select("price").orderBy("price", ascending = False).show(20)